# 01 — Basic Scanpy: Spatial Transcriptomics Analysis

**Spatial Transcriptomics Project | 10x Genomics**

This is the **foundation notebook** of the project. It establishes the core analysis toolkit used in all subsequent notebooks.

**Dataset 1:** Visium — Human Lymph Node (10x Genomics public dataset)  
**Dataset 2:** Synthetic MERFISH-style data (simulated for spatial embedding demo)

**Skills built here (reused in notebooks 02–04):**
- Quality control (QC) of spatial transcriptomics data
- Filtering low-quality cells and genes
- Normalization and log-transformation
- Highly variable gene selection
- PCA → KNN graph → UMAP → Leiden clustering pipeline
- Spatial visualization on tissue images
- Marker gene identification
- Gene expression visualization in spatial context

**Key steps:**
1. Install & import packages
2. Load Visium Human Lymph Node data
3. Quality control & filtering
4. Normalization & dimensionality reduction
5. UMAP & Leiden clustering
6. Spatial visualizations
7. Marker gene analysis
8. Gene expression in spatial context
9. Synthetic MERFISH demo

## 1. Install Dependencies

In [ ]:
!pip install scanpy seaborn openpyxl igraph leidenalg -q

## 2. Import Packages

In [ ]:
from __future__ import annotations

import numpy as np
import anndata
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns

sc.logging.print_versions()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3

## 3. Load Visium Human Lymph Node Dataset

The **Human Lymph Node** dataset is a publicly available 10x Genomics Visium dataset.

**What is a Lymph Node?** A small bean-shaped immune organ that filters lymph fluid and houses immune cells (B cells, T cells, macrophages, etc.). It is an ideal tissue for spatial transcriptomics because it has a highly structured spatial organization — germinal centers, follicles, medulla, and capsule each have distinct cell type compositions.

**AnnData structure:**
- `adata.X` — gene expression matrix (spots × genes)
- `adata.obs` — spot-level metadata (QC metrics)
- `adata.var` — gene-level metadata
- `adata.obsm['spatial']` — 2D spatial coordinates for each spot
- `adata.uns['spatial']` — tissue image data

In [ ]:
# Load the Visium Human Lymph Node dataset
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

# Flag mitochondrial genes (MT- prefix in human data)
# High MT% indicates stressed or dying cells
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

print("AnnData object:")
print(adata)
print(f"\nSpots: {adata.n_obs}")
print(f"Genes: {adata.n_vars}")
print(f"Mitochondrial genes: {adata.var['mt'].sum()}")

## 4. Quality Control (QC)

QC identifies and removes low-quality spots. Key metrics for Visium:

| Metric | What it measures | Problem if too low | Problem if too high |
|---|---|---|---|
| `total_counts` | Total RNA captured per spot | Empty spot or poor capture | Likely doublet or artifact |
| `n_genes_by_counts` | Number of unique genes detected | Low-quality spot | Rare; possible doublet |
| `pct_counts_mt` | % reads from mitochondrial genes | — | Cell stress or death |

**Strategy:** Plot distributions first, then set thresholds based on the data.

> **Save this plot** — QC distribution histograms.

In [ ]:
# Plot QC metric distributions
fig, axs = plt.subplots(1, 4, figsize=(16, 4))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
axs[0].set_title("Total counts (all spots)")
axs[0].set_xlabel("Total counts")

sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"] < 10000],
    kde=False, bins=40, ax=axs[1]
)
axs[1].set_title("Total counts (< 10,000)")
axs[1].set_xlabel("Total counts")

sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[2])
axs[2].set_title("Unique genes per spot")
axs[2].set_xlabel("Genes by counts")

sns.histplot(
    adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 4000],
    kde=False, bins=60, ax=axs[3]
)
axs[3].set_title("Unique genes (< 4,000)")
axs[3].set_xlabel("Genes by counts")

plt.tight_layout()
plt.savefig("scanpy_qc_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_qc_distributions.png")

## 5. Filter Low-Quality Spots and Genes

Based on the QC plots, we apply the following thresholds:
- **min_counts = 5,000**: remove spots with very low total RNA (likely empty)
- **max_counts = 35,000**: remove spots with unusually high counts (possible doublets/artifacts)
- **pct_counts_mt < 20%**: remove spots with high mitochondrial content (dying cells)
- **min_cells = 10**: remove genes detected in fewer than 10 spots (too sparse)

In [ ]:
print(f"Before filtering: {adata.n_obs} spots, {adata.n_vars} genes")

sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)

adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
print(f"After MT filter: {adata.n_obs} spots")

sc.pp.filter_genes(adata, min_cells=10)

print(f"After all filters: {adata.n_obs} spots, {adata.n_vars} genes")

## 6. Normalization & Highly Variable Gene Selection

**Why normalize?** Different spots capture different total amounts of RNA (technical variation). Normalization makes spots comparable.

Steps:
1. **`normalize_total`** — scale each spot so total counts = 10,000 (CPM-like)
2. **`log1p`** — log(x+1) transform to reduce skew and stabilize variance
3. **`highly_variable_genes`** — select the 2,000 genes with most variable expression (biological signal carriers)

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

print(f"Highly variable genes selected: {adata.var['highly_variable'].sum()}")

## 7. Dimensionality Reduction & Clustering

The standard single-cell/spatial pipeline:

```
Raw counts (spots × genes)
    ↓ normalize + log
    ↓ select HVGs
    ↓ PCA (50 components) — linear compression
    ↓ KNN graph — connect similar spots
    ↓ UMAP — 2D non-linear embedding for visualization
    ↓ Leiden — graph-based community detection (clusters)
```

**Leiden** is preferred over k-means because it finds communities in the graph without requiring you to pre-specify the number of clusters.

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(
    adata,
    key_added="clusters",
    flavor="igraph",
    directed=False,
    n_iterations=2
)

print(f"Leiden clusters found: {adata.obs['clusters'].nunique()}")
print("Cluster sizes:")
print(adata.obs["clusters"].value_counts().sort_index())

## 8. UMAP Visualization

UMAP projects the high-dimensional gene expression space into 2D. Spots that are transcriptionally similar cluster together. This gives us a **gene-expression-space** view (no spatial coordinates).

> **Save this plot** — UMAP colored by QC metrics and Leiden clusters.

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)

sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "clusters"],
    wspace=0.4,
    show=False
)
plt.savefig("scanpy_umap_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_umap_clusters.png")

## 9. Spatial Visualization — QC Metrics on Tissue

Now we overlay the same QC information on the actual tissue image. This reveals whether low-quality regions are spatially structured (e.g., edge effects) or random.

> **Save this plot** — spatial view of total counts and gene counts.

In [ ]:
plt.rcParams["figure.figsize"] = (8, 8)

sc.pl.spatial(
    adata,
    img_key="hires",
    color=["total_counts", "n_genes_by_counts"],
    show=False
)
plt.savefig("scanpy_spatial_counts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_spatial_counts.png")

## 10. Spatial Visualization — Leiden Clusters on Tissue

This is the key result: Leiden clusters derived from gene expression, overlaid on the tissue image. This shows **where** each transcriptional state lives in the tissue — connecting molecular identity to tissue anatomy.

> **Save this plot** — clusters on tissue.

In [ ]:
sc.pl.spatial(
    adata,
    img_key="hires",
    color="clusters",
    size=1.5,
    show=False
)
plt.savefig("scanpy_spatial_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_spatial_clusters.png")

## 11. Spatial Visualization — Cropped Region (Clusters 5 & 9)

Cropping allows close inspection of specific tissue regions. Here we zoom into clusters 5 and 9 with transparency (alpha=0.5) to see the tissue underneath.

This is useful for:
- Confirming that clusters match expected tissue structures (e.g., germinal centers, follicles)
- Comparing small adjacent regions with different gene expression profiles

> **Save this plot** — cropped region showing clusters 5 and 9.

In [ ]:
sc.pl.spatial(
    adata,
    img_key="hires",
    color="clusters",
    groups=["5", "9"],
    crop_coord=[7000, 10000, 0, 6000],
    alpha=0.5,
    size=1.3,
    show=False
)
plt.savefig("scanpy_spatial_crop_clusters5_9.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_spatial_crop_clusters5_9.png")

## 12. Marker Gene Analysis

**Goal:** Find genes that best distinguish each cluster from the rest.

**Method:** Wilcoxon rank-sum test (t-test variant) for each gene in each cluster vs all other clusters. The output is a ranked list of marker genes per cluster.

We visualize the top 10 marker genes for **cluster 9** as a heatmap across all clusters. A strong marker gene shows high expression in cluster 9 and low expression elsewhere.

> **Save this plot** — marker gene heatmap for cluster 9.

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")

sc.pl.rank_genes_groups_heatmap(
    adata,
    groups="9",
    n_genes=10,
    groupby="clusters",
    show=False
)
plt.savefig("scanpy_marker_genes_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_marker_genes_heatmap.png")

## 13. Gene Expression in Spatial Context — CR2

**CR2** (Complement Receptor 2 / CD21) is a B cell marker highly expressed in follicular B cells and germinal center B cells. Visualizing its spatial expression alongside cluster labels helps validate cluster identity.

If cluster 9 (or another cluster) shows high CR2 expression and corresponds to a germinal center region spatially, this is biological validation.

> **Save this plot** — clusters + CR2 gene expression on tissue.

In [ ]:
sc.pl.spatial(
    adata,
    img_key="hires",
    color=["clusters", "CR2"],
    show=False
)
plt.savefig("scanpy_spatial_cr2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_spatial_cr2.png")

## 14. Gene Expression in Spatial Context — COL1A2 & SYPL1

- **COL1A2** (Collagen Type I Alpha 2): a structural protein expressed in fibroblasts and stromal cells. High in connective tissue/capsule regions of lymph nodes.
- **SYPL1** (Synaptophysin-Like Protein 1): expressed in certain immune and stromal cells.

Using `alpha=0.7` makes the H&E tissue image visible beneath the gene expression overlay.

> **Save this plot** — COL1A2 and SYPL1 expression on tissue.

In [ ]:
sc.pl.spatial(
    adata,
    img_key="hires",
    color=["COL1A2", "SYPL1"],
    alpha=0.7,
    show=False
)
plt.savefig("scanpy_spatial_col1a2_sypl1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_spatial_col1a2_sypl1.png")

---
## Part 2: Synthetic MERFISH-style Spatial Data

MERFISH (Multiplexed Error-Robust Fluorescence In Situ Hybridization) is another spatial transcriptomics technology, similar in concept to Xenium (notebook 04).

Here we **simulate** MERFISH-style data to demonstrate:
- How spatial transcriptomics data from non-Visium platforms is structured
- That the same Scanpy analysis pipeline works regardless of platform
- The difference between `sc.pl.umap()` (expression space) and `sc.pl.embedding(basis='spatial')` (physical space)

**Simulation parameters:**
- 645 cells, 12,903 genes
- Expression drawn from a Negative Binomial distribution (realistic count data model)
- Random spatial coordinates over a 5000×5000 µm field

In [ ]:
# Simulate MERFISH-style count data
np.random.seed(42)
n_cells, n_genes = 645, 12903

# Negative Binomial distribution simulates realistic RNA count data
X = np.random.negative_binomial(3, 0.3, size=(n_cells, n_genes)).astype(float)

cell_ids = [f"cell_{i}" for i in range(n_cells)]
gene_ids = [f"gene_{i}" for i in range(n_genes)]

adata_merfish = anndata.AnnData(
    X=X,
    obs=pd.DataFrame(index=cell_ids),
    var=pd.DataFrame(index=gene_ids)
)

# Add simulated spatial coordinates
adata_merfish.obsm["spatial"] = np.column_stack([
    np.random.uniform(0, 5000, n_cells),
    np.random.uniform(0, 5000, n_cells)
])

print("Simulated MERFISH AnnData:")
print(adata_merfish)
print(f"\nCells: {adata_merfish.n_obs}")
print(f"Genes: {adata_merfish.n_vars}")
print(f"Spatial coords shape: {adata_merfish.obsm['spatial'].shape}")

## 15. MERFISH: Normalize, Cluster, and Embed

Same pipeline as Visium — demonstrating that the Scanpy toolkit is platform-agnostic. Note `sc.pp.normalize_per_cell` (older API, equivalent to `normalize_total`) is used here for compatibility.

In [ ]:
# Normalize, log, PCA, UMAP, Leiden
sc.pp.normalize_per_cell(adata_merfish, counts_per_cell_after=1e6)
sc.pp.log1p(adata_merfish)
sc.pp.pca(adata_merfish, n_comps=15)
sc.pp.neighbors(adata_merfish)
sc.tl.umap(adata_merfish)
sc.tl.leiden(
    adata_merfish,
    key_added="clusters",
    resolution=0.5,
    n_iterations=2,
    flavor="igraph",
    directed=False
)

print(f"Leiden clusters found: {adata_merfish.obs['clusters'].nunique()}")
print(adata_merfish)

## 16. MERFISH: UMAP Visualization

UMAP of the simulated MERFISH data. Since the data is randomly generated, clusters reflect statistical patterns in the Negative Binomial sampling rather than real biology — but the workflow is identical to real data.

> **Save this plot** — MERFISH UMAP with Leiden clusters.

In [ ]:
sc.pl.umap(
    adata_merfish,
    color="clusters",
    show=False
)
plt.savefig("scanpy_merfish_umap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_merfish_umap.png")

## 17. MERFISH: Spatial Embedding

`sc.pl.embedding(basis='spatial')` uses the physical spatial coordinates (stored in `adata.obsm['spatial']`) instead of UMAP coordinates. This shows cluster distribution in the tissue, not in expression space.

Comparing UMAP vs spatial embedding:
- **UMAP**: similar expression → close together
- **Spatial**: physical proximity → close together

If clusters are spatially coherent (same cluster = same tissue region), expression and spatial organization agree — a hallmark of good spatial transcriptomics data.

> **Save this plot** — MERFISH spatial embedding with Leiden clusters.

In [ ]:
sc.pl.embedding(
    adata_merfish,
    basis="spatial",
    color="clusters",
    show=False
)
plt.savefig("scanpy_merfish_spatial.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scanpy_merfish_spatial.png")

## 18. Summary & Key Findings

### Visium Human Lymph Node

| Step | Result |
|---|---|
| After QC filtering | Retained high-quality tissue-covered spots |
| Leiden clustering | Identified transcriptionally distinct spatial regions |
| Spatial clusters | Correspond to known lymph node structures |
| CR2 expression | Marks B cell follicle regions (germinal centers) |
| COL1A2 expression | Marks stromal/capsular regions |

### What This Notebook Established (used in notebooks 02–04)

| Skill | Used in |
|---|---|
| QC metrics + filtering | 02, 03, 04 |
| normalize → log → HVG | 02, 03, 04 |
| PCA → neighbors → UMAP → Leiden | 02, 03, 04 |
| `sc.pl.spatial` / `sq.pl.spatial_scatter` | 02, 03, 04 |
| Marker gene ranking | (extended in 03 with Moran's I) |
| Gene expression overlay on tissue | 02, 03, 04 |

**Next notebook:** `02_visium_fluorescence.ipynb` — adds image feature extraction, cell segmentation, and multi-channel fluorescence analysis.

In [ ]:
# Save the processed AnnData objects
adata.write_h5ad("scanpy_lymph_node_processed.h5ad")
adata_merfish.write_h5ad("scanpy_merfish_processed.h5ad")

print("Saved processed AnnData files.")
print("\nAll figures to download for GitHub:")
figures = [
    "scanpy_qc_distributions.png",
    "scanpy_umap_clusters.png",
    "scanpy_spatial_counts.png",
    "scanpy_spatial_clusters.png",
    "scanpy_spatial_crop_clusters5_9.png",
    "scanpy_marker_genes_heatmap.png",
    "scanpy_spatial_cr2.png",
    "scanpy_spatial_col1a2_sypl1.png",
    "scanpy_merfish_umap.png",
    "scanpy_merfish_spatial.png",
]
for f in figures:
    print(f"  - {f}")